In [ ]:
print("Thinking...")
import warnings
import pandas as pd                    # For reading CSV files into tables
import numpy as np                     # For numerical calculations
print("Thinking...")
from scipy import signal               # For the Welch PSD / spectral analysis
from matplotlib import pyplot as plt   # For creating plots
import matplotlib.ticker as ticker
import os

warnings.simplefilter('ignore')

###--------------------------------------------------------------------------------------------------------------------------###
'''------------------------------------------------- SECTION 1 — FILE PATH --------------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###


## Set this to the location of your seismic CSV data file.

filepaths = []
filepaths.append(r"C:\Users\cacam\Downloads\Seis07-close-i80-NS-to-i80_2026-04-06T00-32-21-479.csv")
filepaths.append(r"C:\Users\cacam\Downloads\Seis20-cleanHarb-First-hills-by-road-to-AFB_2026-04-06T22-19-33-391.csv")
filepaths.append(r"C:\Users\cacam\Downloads\Seis07-close-i80-NS-to-i80_2026-04-06T00-32-21-479.csv")
filepaths.append(r"C:\Users\cacam\Downloads\Seis010hvVisibleonflats_2026-04-05T08-33-59-274.csv")
filepaths.append(r"C:\Users\cacam\Downloads\Seis010hvVisibleonflats_2026-04-05T08-33-59-274.csv")


distances = []

distances.append("0")
distances.append("0")
distances.append("0")
distances.append("0")
distances.append("0")



###--------------------------------------------------------------------------------------------------------------------------###
'''----------------------------------------------- SECTION 2 — PLOT SETTINGS ------------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###



## Adjust these to control how the output plots look and how the FFT is computed.


# Y-axis (amplitude) range in m/√Hz or m·s⁻¹/√Hz:
y_max = 10e-5                                            # Top of the plot    (higher = more noise visible)
y_min = 10e-14                                           # Bottom of the plot (lower = more detail at quiet levels)


# X-axis (frequency) range in Hz:
x_max = 100                                              # Highest frequency to show
x_min = 0.1                                              # Lowest frequency to show


# FFT (spectral analysis) settings:
fft_length = 8                                           # Length of each FFT window in seconds
                                                         #   Longer = better frequency resolution, less time averaging

overlap    = 50                                          # Overlap between consecutive FFT windows, in percent (0–99)


# Were the pre-amplifiers used during recording?
pre_amps = False                                         # Set to True if pre-amps were active


# Title of the plot
plot_title = "Seismic Displacement Data ASD"



###--------------------------------------------------------------------------------------------------------------------------###
'''------------------------------------------ SECTION 3 — READ THE FIELD DATA FILE ------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###




cmap = plt.get_cmap("tab20")
colors = [cmap(i) for i in range(len(filepaths))]

raw_data_list = []


plt.figure(figsize=(20, 8))  # one figure for all files

for i in np.arange(len(filepaths)):
    metadata = {}
    with open(filepaths[i], 'r') as f:
        for _ in range(5):
            line = f.readline().strip()
            if ':' in line:
                key, value = line.split(':', 1)
                metadata[key.strip()] = value.strip()

    print("\nreading metadata...")
    sr = float(metadata["Sample Rate"])

    
    raw_data = pd.read_csv(filepaths[i], skiprows=5, delimiter=',')
    raw_data.columns = ["Sample", "Time (s)", "Noise", "East", "North", "Zed", "blank"]
    raw_data_list.append(raw_data)

    
    if pre_amps:
        calibration = (0.0125e-1) / 21
    else:
        calibration = 0.0125e-1

        
    data     = raw_data_list[i]["Zed"] * calibration
    distance = distances[i]
    color    = colors[i]
    lable    = str("seis_{0}_Z".format(i))

    
    
###--------------------------------------------------------------------------------------------------------------------------###
    '''--------------------------------- SECTION 4 — ASD CALCULATION AND PLOTTING FUNCTION -------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###



    def welch_asd(data, sr):
        data = np.asarray(data)
        frequencies, power = signal.welch(
            data, sr, window='hamming',
            nperseg = int(sr * fft_length),
            noverlap = int(round(sr * (overlap * 0.01))) )
        
        
        return frequencies, np.sqrt(power)

    freq, amp = welch_asd(data, sr)
    disp = np.where(freq > 0, amp / (2 * np.pi * freq), np.nan)

    
### ----------------------------------------------------- Plotting Data  ----------------------------------------------------###


    plt.plot(freq, disp,
             color      = color,
             linewidth  = 1.75,
             alpha      = 0.5,
             label      = str(lable) + " " + str(distance) + " [km]")


# --- everything below runs once, after all files are plotted ---
plt.yscale('log')
plt.xscale('log')

ax = plt.gca()
ax.minorticks_on()

plt.title(plot_title, fontweight='bold', fontsize=25)
plt.title(f'FFT: {fft_length}s', fontsize=18, loc='left', style='italic')
plt.title(f'Overlap: {overlap}%', fontsize=18, loc='right', style='italic')

plt.xlabel('Frequency [Hz]', fontweight='bold', fontsize=20)
plt.ylabel('Amplitude [m/√Hz]', fontweight='bold', fontsize=20)

plt.yticks(fontsize=20, fontweight='bold')
plt.xticks(fontsize=20, fontweight='bold')


ax.tick_params(axis='both', which='minor', labelsize=16)
for label in ax.get_yticklabels(which='minor'):
    label.set_fontweight('bold')
    
for label in ax.get_xticklabels(which='minor'):
    label.set_fontweight('bold')

    
ax.yaxis.set_major_locator(ticker.LogLocator(base = 10.0, 
                                             numticks = 20))

ax.yaxis.set_minor_locator(ticker.LogLocator(base = 10.0, 
                                             subs = (0.2, 0.4, 0.6, 0.8), 
                                             numticks = 20))

plt.ylim(y_min, y_max)
plt.xlim(x_min, x_max)

plt.legend(loc='lower left', fontsize=18, ncol=2)
plt.grid(True, which='both', ls='-')
plt.tight_layout()
plt.show()